In [1]:
import os

from pyspark.sql.functions import col

from spark_esri import spark_start, spark_stop

In [8]:
extra_options = " ".join([
    "-Djava.awt.headless=true",
    "-XX:+UseG1GC",
    "-XX:+UseCompressedOops",
    "-XX:-AggressiveHeap",
    "-XX:+UseStringDeduplication",
    "-XX:+ParallelRefProcEnabled"])

config = {
    "spark.driver.memory": "16G",
    "spark.driver.extraJavaOptions": extra_options,
    "spark.executor.instances": 1,
    "spark.executor.memory": "16G",
    "spark.executor.extraJavaOptions": extra_options
}

spark_stop()
spark = spark_start(config)

In [9]:
sp_ref = arcpy.SpatialReference(4326)

In [10]:
fields = ["SHAPE@X", "SHAPE@Y"]
with arcpy.da.SearchCursor("Broadcast", fields, spatial_reference=sp_ref) as rows:
    df = spark\
        .createDataFrame(rows, "lon double,lat double")\
        .select((col("lon") + 1.0).alias("lon"), (col("lat") + 1.0).alias("lat"))\
        .cache()

In [11]:
df.count()

1365578

In [12]:
%%time

df\
    .write\
    .parquet(os.path.join("C:", os.sep, "temp", "delete_me.prq"), mode="overwrite")

Wall time: 2.89 s


In [13]:
%%time

ws = "memory"
nm = "DeleteMe"

fc = os.path.join(ws, nm)

arcpy.management.Delete(fc)

arcpy.management.CreateFeatureclass(ws, nm, "POINT", spatial_reference=sp_ref)

with arcpy.da.InsertCursor(fc, ["SHAPE@X", "SHAPE@Y"]) as cursor:
    for row in df.toLocalIterator():
        cursor.insertRow((row["lon"], row["lat"]))


Wall time: 12.1 s
